In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ==========================================
# 1. Geometry & Utils (Geometry & Utils)
# ==========================================

def torus_to_cartesian(theta, phi, R=1.0, r=0.20):
    """
     Torus intrinsic coordinates (theta, phi) to Cartesian coordinates (x, y, z)
    Theta: tube angle (tube cross-section)
    Phi: torus angle (tube overall)
    """
    x = (R + r * np.cos(theta)) * np.cos(phi)
    y = (R + r * np.cos(theta)) * np.sin(phi)
    z = r * np.sin(theta)
    return np.stack([x, y, z], axis=-1)

def get_torus_mesh_data(R=1.0, r=0.20, n_theta=150, n_phi=150):
    """
    Generate Torus 's  Mesh and pre-computed lighting material Facecolors。
    Retains the specular, diffuse, and ambient parameters。
    """
    # 1. Mesh Grid
    theta = np.linspace(0, 2*np.pi, n_theta)
    phi = np.linspace(0, 2*np.pi, n_phi)
    THETA, PHI = np.meshgrid(theta, phi)

    X_surf = (R + r * np.cos(THETA)) * np.cos(PHI)
    Y_surf = (R + r * np.cos(THETA)) * np.sin(PHI)
    Z_surf = r * np.sin(THETA)

    # 2. Lighting Calculation (Custom Shader)
    # Normal vectors
    nx = np.cos(THETA) * np.cos(PHI)
    ny = np.cos(THETA) * np.sin(PHI)
    nz = np.sin(THETA)
    # Normalize (though logically they are already unit for a torus tube center relative)
    # Actually for lighting let's be precise:
    # The normal of the surface is simply the direction from tube center to surface point.
    norm = np.sqrt(nx**2 + ny**2 + nz**2)
    nx, ny, nz = nx/norm, ny/norm, nz/norm

    # Lighting Parameters
    light_dir = np.array([0.5, 0.8, 0.8])
    light_dir = light_dir / np.linalg.norm(light_dir)
    view_dir = np.array([0.3, 0.5, 0.8])
    view_dir = view_dir / np.linalg.norm(view_dir)

    # Diffuse
    diffuse = np.maximum(0, nx*light_dir[0] + ny*light_dir[1] + nz*light_dir[2])

    # Specular (Phong)
    reflect = 2 * (nx*light_dir[0] + ny*light_dir[1] + nz*light_dir[2])[:, :, np.newaxis] * \
              np.stack([nx, ny, nz], axis=-1) - light_dir
    reflect_dot_view = np.maximum(0, reflect[:, :, 0]*view_dir[0] + 
                                  reflect[:, :, 1]*view_dir[1] + 
                                  reflect[:, :, 2]*view_dir[2])
    specular = reflect_dot_view ** 20 * 0.3

    # Combine
    ambient = 0.55
    intensity = ambient + (1 - ambient) * diffuse * 1.0 + specular
    intensity = np.clip(intensity, 0, 1)

    # Apply Color
    base_color = np.array([0.82, 0.91, 0.91]) # Light Cyan-ish
    facecolors = np.zeros((*intensity.shape, 4))
    for i in range(3):
        facecolors[:, :, i] = base_color[i] * intensity
    facecolors[:, :, 3] = 1.0 # Alpha

    return X_surf, Y_surf, Z_surf, facecolors

def create_sphere(center, radius, color, n=20):
    """
    Create a lit sphere for path decoration。
    """
    light_dir = np.array([0.5, 0.8, 0.8])
    light_dir = light_dir / np.linalg.norm(light_dir)
    view_dir = np.array([0.3, 0.5, 0.8])
    view_dir = view_dir / np.linalg.norm(view_dir)
    
    u = np.linspace(0, 2*np.pi, n)
    v = np.linspace(0, np.pi, n)
    U, V = np.meshgrid(u, v)
    
    x = center[0] + radius * np.cos(U) * np.sin(V)
    y = center[1] + radius * np.sin(U) * np.sin(V)
    z = center[2] + radius * np.cos(V)
    
    nx = np.cos(U) * np.sin(V)
    ny = np.sin(U) * np.sin(V)
    nz = np.cos(V)
    
    diffuse = np.maximum(0, nx*light_dir[0] + ny*light_dir[1] + nz*light_dir[2])
    
    reflect = 2 * (nx*light_dir[0] + ny*light_dir[1] + nz*light_dir[2])[:, :, np.newaxis] * \
              np.stack([nx, ny, nz], axis=-1) - light_dir
    reflect_dot_view = np.maximum(0, reflect[:, :, 0]*view_dir[0] + 
                                   reflect[:, :, 1]*view_dir[1] + 
                                   reflect[:, :, 2]*view_dir[2])
    specular = reflect_dot_view ** 30 * 0.5
    
    sphere_intensity = 0.3 + 0.7 * diffuse + specular
    sphere_intensity = np.clip(sphere_intensity, 0, 1)
    
    base = np.array(color)
    sphere_colors = np.zeros((*sphere_intensity.shape, 4))
    for i in range(3):
        sphere_colors[:, :, i] = base[i] * sphere_intensity
    sphere_colors[:, :, 3] = 1.0
    
    return x, y, z, sphere_colors

# ==========================================
# 2. Data Generation (OOD Simulation)
# ==========================================

def generate_full_torus_data(n_samples=2000, R=1.0, r=0.20, noise_std=0.015, seed=42):
    """
    GenerateCover entire torus distribution。
    """
    np.random.seed(seed)
    
    phi = np.random.uniform(0, 2*np.pi, n_samples)
    theta = np.random.uniform(0, 2*np.pi, n_samples)
    
    pts = torus_to_cartesian(theta, phi, R, r)
    pts += np.random.normal(0, noise_std, pts.shape)
    
    return pts

def generate_twin_peaks_data(n_samples=2000, x_range=(-1.0, 1.0), y_range=(-1.0, 1.0), 
                             peak_height=0.5, peak_width=0.5, noise_std=0.01, seed=42):
    """
    Generatedistributed on bimodal surface z = f(x, y) onsample data。
    """
    np.random.seed(seed)
    
    # 1. 在plane上均匀sampling (x, y)
    x = np.random.uniform(x_range[0], x_range[1], n_samples)
    y = np.random.uniform(y_range[0], y_range[1], n_samples)
    
    # 2. 定义两个峰's 中心 (挡在path中间)
    p1 = np.array([0.0, 0.0])

    
    # 3. compute高度 z (双峰高斯和)
    h1 = peak_height * np.exp(-((x - p1[0])**2 + (y - p1[1])**2) / (2 * peak_width**2))
    z = h1 
    
    # 4. 组合成 3D 点云并添加少量噪声
    pts = np.stack([x, y, z], axis=1)
    pts += np.random.normal(0, noise_std, pts.shape)
    
    return pts.astype(np.float32)
# ==========================================
# 3. Main Visualization (Main Visualization)
# ==========================================
def visualize_twin_peaks(pts, peak_height=0.5, peak_width=0.5):
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    # 1. 绘制背景Surface mesh (为了看清山峰形状)
    x_grid = np.linspace(-1.1, 1.1, 50)
    y_grid = np.linspace(-1.1, 1.1, 50)
    X, Y = np.meshgrid(x_grid, y_grid)
    
    p1= [0.0, 0.0]
    Z = peak_height * (np.exp(-((X - p1[0])**2 + (Y - p1[1])**2) / (2 * peak_width**2)))
    
    # # 's 
    surf = ax.plot_surface(X, Y, Z, cmap='terrain', alpha=0.3, linewidth=0, antialiased=True)
    
    # 2. 绘制Generate's 数据点
    ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=2, c='royalblue', alpha=0.5, label='Training Data')
    
    # 3. 标注起点和终点 (演示path对比时使用)
    ax.scatter([-1.2], [0], [0], color='red', s=100, marker='^', label='Start (-1.2, 0)')
    ax.scatter([1.2], [0], [0], color='green', s=100, marker='v', label='End (1.2, 0)')

    ax.set_title("Twin Peaks Manifold Dataset")
    ax.set_xlabel("X (Width)")
    ax.set_ylabel("Y (Length)")
    ax.set_zlabel("Z (Height)")
    ax.set_zlim(0, 1.0)
    ax.legend()
    
    plt.show()

# # 
data = generate_twin_peaks_data(2000)
visualize_twin_peaks(data)
def plot_torus_experiment(
    R=1.0, r=0.20,
    paths=None,            # List of dicts: [{'path': array, 'color': 'cyan', 'style': 'solid'}, ...]
    training_data=None,    # (N, 3) array of black dots
    landmarks=None,        # List of coordinates for big purple spheres (Start/End)
    output_path='torus_experiment.png',
    view_elev=25, view_azim=-30
):
    """
    Generic Torus experiment plotting function。
    """
    # 1. Setup Environment
    fig = plt.figure(figsize=(12, 10), facecolor='white')
    ax = fig.add_subplot(111, projection='3d', computed_zorder=False) # Important for manual layering
    ax.set_facecolor('white')
    
    # 2. Render Torus Surface
    X_surf, Y_surf, Z_surf, facecolors = get_torus_mesh_data(R, r)
    ax.plot_surface(X_surf, Y_surf, Z_surf,
                   facecolors=facecolors,
                   edgecolor='none',
                   antialiased=True,
                   shade=False,
                   rcount=100, ccount=100,
                   zorder=0) # Background
    
    # 3. Render Training Data (if provided)
    if training_data is not None:
        ax.scatter(training_data[:, 0], training_data[:, 1], training_data[:, 2],
                   s=1.5, c='black', alpha=0.15, zorder=1, label='Training Data')

    # 4. Render Paths
    # Sphere colors mapping
    color_map = {
        'cyan': [0.0, 0.75, 0.75],
        'gray': [0.6, 0.6, 0.6],
        'red':  [0.8, 0.2, 0.2],
        'blue': [0.1, 0.3, 0.8]
    }
    
    sphere_radius = 0.025
    
    if paths:
        for idx, path_info in enumerate(paths):
            path_pts = path_info['path']
            c_name = path_info.get('color', 'gray')
            c_rgb = color_map.get(c_name, color_map['gray'])
            
            # Draw Line
            ax.plot(path_pts[:, 0], path_pts[:, 1], path_pts[:, 2],
                    color='k', linewidth=1.5, zorder=5 + idx)
            
            # Draw Spheres along path (downsample slightly for visual clarity)
            # interval = max(1, len(path_pts) // 15)
            # Or use explicit step if provided, else every 2nd point
            step = path_info.get('sphere_step', 2)
            
            for pt in path_pts[1:-1:step]: # Skip start/end to avoid clash with landmarks
                sx, sy, sz, sc = create_sphere(pt, sphere_radius, c_rgb, n=12)
                ax.plot_surface(sx, sy, sz, facecolors=sc, edgecolor='none',
                                shade=False, zorder=10 + idx, antialiased=True)

    # 5. Render Landmarks (Start/End)
    if landmarks:
        landmark_color = [0.6, 0.3, 0.8] # Purple
        for pt in landmarks:
            sx, sy, sz, sc = create_sphere(pt, sphere_radius * 1.5, landmark_color, n=18)
            ax.plot_surface(sx, sy, sz, facecolors=sc, edgecolor='none',
                           shade=False, zorder=50, antialiased=True)

    # 6. Final Polish
    limit = R + r + 0.1
    ax.set_xlim([-limit, limit])
    ax.set_ylim([-limit, limit])
    ax.set_zlim([-0.4, 0.4])
    ax.set_box_aspect([1, 1, 0.35])
    
    ax.set_axis_off()
    ax.view_init(elev=view_elev, azim=view_azim)
    
    plt.tight_layout()

    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white', edgecolor='none')
    print(f"Saved visualization to {output_path}")
    plt.show()
    plt.close()


# ==========================================
# 4. Run Experiment (Example Usage)
# ==========================================

    
# 1. Define geometry parameters
R_val, r_val = 1.2, 0.40

# 2. Generate Patchy Data (OOD Scenario)
data_points = generate_full_torus_data(n_samples=10000, R=R_val, r=r_val, noise_std=0.005)

# 3. Define path (Simulate your algorithm vs Baseline)
n_steps = 40

# A. Your method (Geodesic on Manifold) (Geodesic on Manifold): Walk along torus surface，even without data in between
phi_geo = np.linspace(0, np.pi, n_steps)
theta_geo = np.zeros(n_steps) # Stay on equator
path_ours = torus_to_cartesian(theta_geo, phi_geo, R_val, r_val)

# B. Baseline (Linear/Euclidean): Go straight through the hole
start_pt = path_ours[0]
end_pt = path_ours[-1]
t = np.linspace(0, 1, n_steps)
path_baseline = np.array([start_pt + ti * (end_pt - start_pt) for ti in t])

# 4. Pack path information
path_list = [
    {
        'path': path_ours,
        'color': 'cyan',
        'sphere_step': 2
    },
    {
        'path': path_baseline,
        'color': 'gray',
        'sphere_step': 2
    }
]

# 5. Define start/end points (purple spheres)
landmarks_pts = [start_pt, end_pt]

# 6. One-click plotting
plot_torus_experiment(
    R=R_val, r=r_val,
    paths=path_list,
    training_data=data_points,  # Pass in data points
    landmarks=landmarks_pts,
    output_path='output/torus2_ood_experiment.png',
    view_elev=35, # Top-down view makes it easier to see the gap Gap
    view_azim=-80
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


class PositionalEmbedding(nn.Module):
    """Sinusoidal positional encoding for noise level"""

    def __init__(self, dim=128):
        super().__init__()
        self.dim = dim

    def forward(self, sigma):
        """sigma: [B] -> embedding: [B, dim]"""
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=sigma.device) * -emb)
        emb = sigma[:, None] * emb[None, :]
        emb = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return emb


class SimpleMLP(nn.Module):
    """SimpleMLPbackbone network"""

    def __init__(self, data_dim=2, hidden_dim=128, num_layers=4):
        super().__init__()
        self.sigma_embed = PositionalEmbedding(hidden_dim)

        self.input_proj = nn.Linear(data_dim, hidden_dim)

        self.layers = nn.ModuleList(
            [
                nn.Sequential(nn.Linear(hidden_dim * 2, hidden_dim), nn.SiLU())
                for _ in range(num_layers)
            ]
        )

        self.output_proj = nn.Linear(hidden_dim, data_dim)
        # Initialize output layer to0
        nn.init.zeros_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, x, sigma_emb):
        """x: [B, 2], sigma_emb: [B, 128] -> output: [B, 2]"""
        h = self.input_proj(x)

        for layer in self.layers:
            h_concat = torch.cat([h, sigma_emb], dim=-1)
            h = layer(h_concat) + h  # residual connection

        return self.output_proj(h)


class EDMDenoiser(nn.Module):
    """EDM X-pred Denoiser: Directly predict clean data x_0"""

    def __init__(self, data_dim=2, hidden_dim=128, num_layers=4, sigma_data=1.0):
        super().__init__()
        self.sigma_data = sigma_data
        self.backbone = SimpleMLP(data_dim, hidden_dim, num_layers)

    def forward(self, x_noisy, sigma):
        """
        EDMPreconditioning：Input/output undergo special scaling
        x_noisy: [B, 2] noisy data
        sigma: [B] noise level
        return: [B, 2] predicted clean data x_0
        """
        if sigma.ndim == 2:
            sigma = sigma.squeeze(-1)

        # EDMPreconditioning coefficients（Core！）
        c_skip = self.sigma_data**2 / (sigma**2 + self.sigma_data**2)
        c_out = sigma * self.sigma_data / torch.sqrt(sigma**2 + self.sigma_data**2)
        c_in = 1 / torch.sqrt(self.sigma_data**2 + sigma**2)
        c_noise = torch.log(sigma) / 4

        # Scale input
        x_scaled = c_in[:, None] * x_noisy

        # Embed noise level
        sigma_emb = self.backbone.sigma_embed(c_noise)

        # Network predicts residual
        F_x = self.backbone(x_scaled, sigma_emb)

        # Preconditioned output = skip connection + scaled residual
        x_0_pred = c_skip[:, None] * x_noisy + c_out[:, None] * F_x

        return x_0_pred


# Test network
model = EDMDenoiser(data_dim=3, hidden_dim=128, num_layers=4)
x_test = torch.randn(4, 3)
sigma_test = torch.rand(4) * 10
output = model(x_test, sigma_test)
print(f"Input shape: {x_test.shape}")
print(f"Output shape: {output.shape}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class EDMLoss(nn.Module):
    """
    EDMTraining Loss：
    - fromlog-normaldistribution sample noise level sigma
    - Add noise：x_noisy = x_0 + sigma * noise
    - Let network predict x_0
    - Use adaptive weights
    """

    def __init__(self, P_mean=-1.2, P_std=1.2, sigma_data=1.0):
        super().__init__()
        self.P_mean = P_mean
        self.P_std = P_std
        self.sigma_data = sigma_data

    def forward(self, denoiser, x_0):
        """
        x_0: [B, 2] clean data
        return: scalar loss
        """
        B = x_0.shape[0]

        # Sample noise level：sigma ~ LogNormal(P_mean, P_std^2)
        rnd_normal = torch.randn(B, device=x_0.device)
        sigma = (rnd_normal * self.P_std + self.P_mean).exp()

        # Add Gaussian noise
        noise = torch.randn_like(x_0)
        x_noisy = x_0 + sigma[:, None] * noise

        # Network predicts clean data
        x_0_pred = denoiser(x_noisy, sigma)

        # EDMAdaptive weights
        weight = (sigma**2 + self.sigma_data**2) / (sigma * self.sigma_data) ** 2

        # WeightedMSEloss
        loss = weight[:, None] * (x_0_pred - x_0) ** 2
        return loss.mean()


def train_edm(model, data, num_epochs=1000, batch_size=256, lr=1e-3, device="cpu"):
    """Training function"""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = EDMLoss()

    dataset = torch.utils.data.TensorDataset(data)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    losses = []

    for epoch in tqdm(range(num_epochs)):
        epoch_loss = 0
        for (x_0,) in loader:
            x_0 = x_0.to(device)

            loss = criterion(model, x_0)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}, Loss: {avg_loss:.6f}")

    return losses


# Train model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

train_data = generate_full_torus_data(n_samples=20000,noise_std=0.005, R=R_val, r=r_val)
train_data = torch.from_numpy(train_data).float().to(device)
model = EDMDenoiser(data_dim=3, hidden_dim=128, num_layers=4)

losses = train_edm(
    model, train_data, batch_size=2048, num_epochs=1000, lr=1e-3, device=device
)

# Visualize training curve
plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.yscale("log")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
@torch.no_grad()
def edm_sampler(
    denoiser,
    batch_size,
    data_dim=3,
    num_steps=18,
    sigma_min=0.002,
    sigma_max=80.0,
    rho=7.0,
    device="cpu",
):
    """
    EDMSampler（2orderHeunmethod）
    From pure noise -> clean data
    """
    denoiser.eval()

    # time stepsDiscretize（Power-law schedule）
    step_indices = torch.arange(num_steps, device=device, dtype=torch.float32)
    t_steps = (
        sigma_max ** (1 / rho)
        + step_indices
        / (num_steps - 1)
        * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
    ) ** rho
    t_steps = torch.cat([t_steps, torch.zeros(1, device=device)])

    # fromGaussian noise initialization
    x = torch.randn(batch_size, data_dim, device=device) * t_steps[0]

    # # 
    for i in range(num_steps):
        sigma_cur = t_steps[i]
        sigma_next = t_steps[i + 1]

        # === 1order Euler 步 ===
        # # clean data
        x_0_pred = denoiser(x, sigma_cur.repeat(batch_size))

        # # 
        d = (x - x_0_pred) / sigma_cur

        # # noise level
        x_next = x + (sigma_next - sigma_cur) * d

        # === 2order Heun 校正 ===
        if i < num_steps - 1:
            # # 
            x_0_pred_next = denoiser(x_next, sigma_next.repeat(batch_size))
            d_next = (x_next - x_0_pred_next) / sigma_next

            # # 's 
            x = x + (sigma_next - sigma_cur) * (0.5 * d + 0.5 * d_next)
        else:
            x = x_next

    return x


# Generate样本
num_samples = 10000
generated = edm_sampler(
    model, batch_size=num_samples, num_steps=18, device=device
).cpu()

print(f"Generated shape: {generated.shape}")
print(f"Generated range: [{generated.min():.2f}, {generated.max():.2f}]")

In [ ]:
plot_torus_experiment(
    R=R_val, r=r_val,
    paths=path_list,
    training_data=generated.numpy(),  # Pass in data points
    landmarks=landmarks_pts,
    output_path='output/torus3_ood_experiment_gen.png',
    view_elev=35, # Top-down view makes it easier to see the gap Gap
    view_azim=-80
)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.func import grad, jvp, vmap, jacrev


# Enable high precision for stable geometry computation
# torch.set_default_dtype(torch.float64)

class TorusGeometry:
    def __init__(self, R=1.2, r=0.4):
        self.R = R
        self.r = r

    def distance(self, x):
        """Signed distance to torus surface."""
        rho = torch.sqrt(x[..., 0]**2 + x[..., 1]**2)
        return torch.sqrt((rho - self.R)**2 + x[..., 2]**2) - self.r

    def frame(self, x):
        """Get orthonormal frame (e_theta, e_phi, n)."""
        # Ensure x has batch dim for safe norms
        if x.ndim == 1: x = x.unsqueeze(0)
       
        rho = torch.maximum(torch.sqrt(x[:, 0]**2 + x[:, 1]**2), torch.tensor(1e-8))
       
        # e_phi: Toroidal direction
        e_phi = torch.stack([-x[:, 1]/rho, x[:, 0]/rho, torch.zeros_like(rho)], dim=-1)
       
        # Center of tube
        cx, cy = self.R * x[:, 0] / rho, self.R * x[:, 1] / rho
        dx, dy, dz = x[:, 0] - cx, x[:, 1] - cy, x[:, 2]
        tube_dist = torch.maximum(torch.sqrt(dx**2 + dy**2 + dz**2), torch.tensor(1e-8))
       
        # Normal direction
        n = torch.stack([dx/tube_dist, dy/tube_dist, dz/tube_dist], dim=-1)
       
        # Poloidal direction
        e_theta = torch.cross(n, e_phi, dim=-1)
        e_theta = e_theta / torch.maximum(torch.linalg.norm(e_theta, dim=-1, keepdim=True), torch.tensor(1e-8))
       
        return e_theta.squeeze(), e_phi.squeeze(), n.squeeze()

    def metric(self, x, alpha, beta, gamma):
        """Construct metric matrix G(x)."""
        e_theta, e_phi, n = self.frame(x)
        # Handle batch or single point
        if x.ndim == 1:
            outer = lambda u, v: torch.outer(u, v)
        else:
            outer = lambda u, v: torch.einsum('bi,bj->bij', u, v)
           
        return (alpha * outer(e_theta, e_theta) +
                beta  * outer(e_phi, e_phi) +
                gamma * outer(n, n))

class RGFMSolver(nn.Module):
    def __init__(self, score_net, alpha=1.0, beta=1.0, gamma=10.0, lam=1.0):
        super().__init__()
        self.score_net = score_net  # trained score network
        self.geo = TorusGeometry()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.lam = lam
        self.dtype = torch.float32

    def compute_residual_pointwise(self, x, v, a, sigma):
        """increase sigma parameters"""
        # 1. Metric and Inverse
        G = self.geo.metric(x, self.alpha, self.beta, self.gamma)
        G_inv = torch.linalg.inv(G)

        # 2. Christoffel term (unchanged)
        def metric_times_v(z):
            return self.geo.metric(z, self.alpha, self.beta, self.gamma) @ v
        _, term_A = jvp(metric_times_v, (x,), (v,))
       
        def kinetic_energy(z):
            G_z = self.geo.metric(z, self.alpha, self.beta, self.gamma)
            return 0.5 * v @ G_z @ v
        term_B = grad(kinetic_energy)(x)
       
        gamma_vv = G_inv @ (term_A - term_B)
        a_cov = a + gamma_vv

        # 3. Score-based force（Key modification！）
        # score_net Input: (x, sigma) -> score
        with torch.no_grad():
            x_pred = self.score_net(x.unsqueeze(0), sigma.unsqueeze(0)).squeeze(0)
            score = (x_pred - x) / sigma**2
        riem_score = G_inv @ score
       
        # 4. Residual: a + Γ - λ g⁻¹ s = 0
        residual = a_cov + self.lam * riem_score
       
        return residual @ G @ residual

    def compute_loss(self, interior, x0, xN, sigma):
        """sigma: can be scalar or vary along path's 向量"""
        path = torch.cat([x0.unsqueeze(0), interior, xN.unsqueeze(0)], dim=0)
        n = path.shape[0]
        dt = 1.0 / (n - 1)

        v = (path[2:] - path[:-2]) / (2 * dt)
        a = (path[2:] - 2*path[1:-1] + path[:-2]) / (dt**2)
        x_inner = path[1:-1]
       
        # sigma 处理：
        # #  A:  sigma（）
        if sigma.dim() == 0:
            sigmas = sigma.expand(x_inner.shape[0])
        # #  B: 's  sigma（from）
        else:
            sigmas = sigma
       
        # vmap 需要所有Input有相同's  batch 维度
        compute_batch = vmap(self.compute_residual_pointwise)
        losses = compute_batch(x_inner, v, a, sigmas)
       
        return torch.mean(losses)

    def solve(self, x0, xN, sigma, num_points=20, num_iters=200, lr=0.01):
        """
        sigma: score network 's noise level
               - 固定值: torch.tensor(0.1)
               - 或者 schedule: from大到小
        """
        # --- 1. compute起点和终点intrinsic coordinates ---
        def get_intrinsic(x):
            rho = torch.norm(x[:2])
            phi = torch.atan2(x[1], x[0])
            theta = torch.atan2(x[2], rho - self.geo.R)
            return theta, phi
        x0 = x0.to(self.dtype)
        xN = xN.to(self.dtype)
        sigma = sigma.to(self.dtype)
        th0, ph0 = get_intrinsic(x0)
        thN, phN = get_intrinsic(xN)

        # --- 2. Interpolate in angle space（Initialize along manifold）---
        t = torch.linspace(0, 1, num_points, device=x0.device)
        theta_interp = th0 + t * (thN - th0)
        phi_interp = ph0 + t * (phN - ph0)
       
        # --- 3. Map back to 3D space ---
        R, r = self.geo.R, self.geo.r
        x_init = (R + r * torch.cos(theta_interp)) * torch.cos(phi_interp)
        y_init = (R + r * torch.cos(theta_interp)) * torch.sin(phi_interp)
        z_init = r * torch.sin(theta_interp)
        initial_path = torch.stack([x_init, y_init, z_init], dim=1)

        # --- 4. optimization ---
        interior = nn.Parameter(initial_path[1:-1].clone())
        optimizer = torch.optim.Adam([interior], lr=lr)
       
        loss_history = []
       
        print(f"Starting optimization with {num_iters} iterations...")
        for i in range(num_iters):
            optimizer.zero_grad()
           
            # #  sigma！
            loss = self.compute_loss(interior, x0, xN, sigma)
           
            loss.backward()
            torch.nn.utils.clip_grad_norm_([interior], 10.0)
            optimizer.step()
            loss_history.append(loss.item())
           
            if i % 300 == 0:
                print(f"Iter {i:4d} | Loss: {loss.item():.6f}")
               
        final_path = torch.cat([x0.unsqueeze(0), interior.detach(), xN.unsqueeze(0)], dim=0)
        return final_path, loss_history

# ==========================================
# Example Usage
# ==========================================
device = "cuda" if torch.cuda.is_available() else "cpu"
solver_A = RGFMSolver(alpha=.05, beta=.5, gamma=1.0, lam=1.0, score_net=model)

x0 = torch.tensor([1.2, 0.0, 0.4]).to(device)
xN = torch.tensor([-1.2, 0.0, 0.4]).to(device)
sigma = torch.tensor(0.1).to(device)
path_A, losses = solver_A.solve(x0, xN, sigma,num_points=20, num_iters=1501, lr=1e-3)

print("Optimization Done.")

print(path_A.shape)

path_list = [
    {
        'path': path_A.reshape(-1, 3).cpu().numpy(),
        'color': 'cyan',
        'sphere_step': 2
    },
]
plot_torus_experiment(
    R=R_val, r=r_val,
    paths=path_list,
    training_data=generated.numpy(),  # Pass in data points
    landmarks=[x0.cpu().numpy(), xN.cpu().numpy()],
    output_path='output/torus3_experiment_gen_el.png',
    view_elev=35, # Top-down view makes it easier to see the gap Gap
    view_azim=-80
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# ==========================================
# 0. Global Configuration：Publication-quality style
# ==========================================
def set_pub_style():
    plt.rcParams.update({
        'font.family': 'serif',
        'font.serif': ['DejaVu Serif', 'Times New Roman'],
        # 'mathtext.fontset': 'cm',  # Remove this or use 'dejavuserif'
        'mathtext.fontset': 'dejavuserif',
        'axes.labelsize': 14,
        'axes.titlesize': 16,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'figure.dpi': 300,
        'axes.grid': True,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'text.usetex': False,  # Explicitly disable LaTeX
    })

# Helper functions：Create gradient line segments
def get_gradient_segments(x, y, z=None):
    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    if z is not None:
        points_3d = np.array([x, y, z]).T.reshape(-1, 1, 3)
        segments_3d = np.concatenate([points_3d[:-1], points_3d[1:]], axis=1)
        return segments, segments_3d
    return segments

def plot_torus_3d(path, R, r, x0=None, xN=None, filename=None):
    set_pub_style()
    path = np.array(path)
    t = np.linspace(0, 1, len(path))
    cmap = plt.get_cmap('plasma')

    # Create independent Figure
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')

    # Generate Torus Surface mesh
    theta = np.linspace(0, 2 * np.pi, 60)
    phi = np.linspace(0, 2 * np.pi, 30)
    THETA, PHI = np.meshgrid(theta, phi)
    X_s = (R + r * np.cos(PHI)) * np.cos(THETA)
    Y_s = (R + r * np.cos(PHI)) * np.sin(THETA)
    Z_s = r * np.sin(PHI)

    # # 
    ax.plot_surface(X_s, Y_s, Z_s, color='whitesmoke', alpha=0.3,
                      rstride=2, cstride=2, edgecolor='gray', linewidth=0.05,
                      shade=True, antialiased=True)

    # # 3D
    _, segments_3d = get_gradient_segments(path[:,0], path[:,1], path[:,2])
    lc = Line3DCollection(segments_3d, cmap=cmap, norm=plt.Normalize(0, 1), alpha=0.95)
    lc.set_array(t)
    lc.set_linewidth(3)
    ax.add_collection(lc)

    # # 
    if x0 is not None: ax.scatter(*x0, color=cmap(0.0), s=100, marker='o', ec='w', lw=1.5, zorder=10, label='Start')
    if xN is not None: ax.scatter(*xN, color=cmap(1.0), s=140, marker='*', ec='w', lw=1.5, zorder=10, label='End')

    # # 
    limit = R + r + 0.5
    ax.set_xlim(-limit, limit); ax.set_ylim(-limit, limit); ax.set_zlim(-limit, limit)
    ax.set_box_aspect([1,1,0.7])
    ax.axis('off') # # 
    ax.view_init(elev=35, azim=-45)
    ax.set_title(r"3D Trajectory Overview", fontsize=16)
    
    # # 's  Colorbar
    cbar = plt.colorbar(lc, ax=ax, pad=0.1, shrink=0.6, aspect=20)
    cbar.set_label(r'Normalized Time ($t/T_{end}$)', fontsize=12)

    if filename: plt.savefig(filename, bbox_inches='tight',dpi=600,transparent=True)
    plt.show()


# --- Figure 2: Poloidal Projection ---
def plot_poloidal(path, R, r, x0=None, xN=None, filename=None):
    set_pub_style()
    path = np.array(path)
    t = np.linspace(0, 1, len(path))
    cmap = plt.get_cmap('plasma')

    rho = np.sqrt(path[:,0]**2 + path[:,1]**2)
    radial_dev = rho - R
    z_vals = path[:,2]

    # Create independent Figure
    fig, ax = plt.subplots(figsize=(7, 6))

    # # 
    tube_circle = Circle((0, 0), r, color='#555555', fill=False, linestyle='-', linewidth=2)
    tube_bg = Circle((0, 0), r, color='#f5f5f5', fill=True, alpha=0.5)
    ax.add_artist(tube_bg)
    ax.add_artist(tube_circle)
    ax.axhline(0, c='k', lw=1, ls=':', alpha=0.2)
    ax.axvline(0, c='k', lw=1, ls=':', alpha=0.2)

    # # 
    segments = get_gradient_segments(radial_dev, z_vals)
    lc = LineCollection(segments, cmap=cmap, norm=plt.Normalize(0, 1), alpha=0.9)
    lc.set_array(t)
    lc.set_linewidth(3)
    ax.add_collection(lc)

    # # 
    if x0 is not None:
        ax.scatter(np.sqrt(x0[0]**2+x0[1]**2)-R, x0[2], color=cmap(0.0), s=100, ec='w', lw=1.5, marker='o', zorder=5)
    if xN is not None:
        ax.scatter(np.sqrt(xN[0]**2+xN[1]**2)-R, xN[2], color=cmap(1.0), s=140, ec='w', lw=1.5, marker='*', zorder=5)

    # # 
    ax.set_aspect('equal')
    slice_limit = r * 1.3
    ax.set_xlim(-slice_limit, slice_limit)
    ax.set_ylim(-slice_limit, slice_limit)
    ax.set_xlabel(r'Radial Deviation $\rho - R$ [m]')
    ax.set_ylabel(r'Vertical Position $z$ [m]')
    ax.set_title(r"Poloidal Cross-Section", fontsize=16)

    # ax.text(0, r*1.08, 'Top', ha='center', va='bottom', fontsize=10, color='gray')
    # ax.text(-r*1.08, 0, 'Inboard', ha='right', va='center', fontsize=10, color='gray')
    # ax.text(r*1.08, 0, 'Outboard', ha='left', va='center', fontsize=10, color='gray')
    
    # # 's  Colorbar
    cbar = plt.colorbar(lc, ax=ax)
    cbar.set_label(r'Normalized Time ($t/T_{end}$)', fontsize=12)
    if filename: plt.savefig(filename, bbox_inches='tight',dpi=600,transparent=True)
    plt.show()


# --- Figure 3: Toroidal Projection ---
def plot_toroidal(path, R, r, x0=None, xN=None, filename=None):
    set_pub_style()
    path = np.array(path)
    t = np.linspace(0, 1, len(path))
    cmap = plt.get_cmap('plasma')

    # Create independent Figure
    fig, ax = plt.subplots(figsize=(7, 6))

    # #  ( Wedge )
    ax.add_artist(Circle((0,0), R+r, color='#999999', fill=False, lw=1.2, ls='-'))
    ax.add_artist(Circle((0,0), R-r, color='#999999', fill=False, lw=1.2, ls='-'))
    ax.add_artist(Circle((0,0), R, color='gray', lw=1, ls=':', alpha=0.5))
    torus_ring = Wedge((0,0), R+r, 0, 360, width=2*r, color='#e0e0e0', alpha=0.4)
    ax.add_artist(torus_ring)

    # # 
    segments = get_gradient_segments(path[:,0], path[:,1])
    lc = LineCollection(segments, cmap=cmap, norm=plt.Normalize(0, 1), alpha=0.9, zorder=5)
    lc.set_array(t)
    lc.set_linewidth(3)
    ax.add_collection(lc)

    # # 
    if x0 is not None: ax.scatter(x0[0], x0[1], color=cmap(0.0), s=100, ec='w', lw=1.5, marker='o', zorder=10)
    if xN is not None: ax.scatter(xN[0], xN[1], color=cmap(1.0), s=140, ec='w', lw=1.5, marker='*', zorder=10)

    # # 
    ax.set_aspect('equal')
    top_limit = R + r + 0.5
    ax.set_xlim(-top_limit, top_limit)
    ax.set_ylim(-top_limit, top_limit)
    ax.set_xlabel(r'$x$ [m]')
    ax.set_ylabel(r'$y$ [m]')
    ax.set_title(r"Toroidal (Top-Down) Projection", fontsize=16)
    
    # # 's  Colorbar
    cbar = plt.colorbar(lc, ax=ax)
    cbar.set_label(r'Normalized Time ($t/T_{end}$)', fontsize=12)

    if filename: plt.savefig(filename, bbox_inches='tight',dpi=600,transparent=True)
    plt.show()
# ==========================================
path_np_A = path_A.detach().cpu().numpy()
x0_np = x0.detach().cpu().numpy()
xN_np = xN.detach().cpu().numpy()
file_base_A = 'output/torus_st8_A'
plot_torus_3d(path_np_A, R_val, r_val, x0_np, xN_np, filename=file_base_A+'_3d.png')
plot_poloidal(path_np_A, R_val, r_val, x0_np, xN_np, filename=file_base_A+'_polo.png')
plot_toroidal(path_np_A, R_val, r_val, x0_np, xN_np, filename=file_base_A+'_tor.png')


# visualize_torus_precise()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
solver_B = RGFMSolver(alpha=5.0, beta=.1, gamma=1.0, lam=1.0, score_net=model)

x0 = torch.tensor([1.2, 0.0, 0.4]).to(device)
xN = torch.tensor([-1.2, 0.0, 0.4]).to(device)
sigma = torch.tensor(0.10).to(device)
path_B, losses = solver_B.solve(x0, xN, sigma,num_points=20, num_iters=1501, lr=5e-3)

print("Optimization Done.")

print(path_B.shape)

path_list = [
    {
        'path': path_B.reshape(-1, 3).cpu().numpy(),
        'color': 'cyan',
        'sphere_step': 2
    },
]
plot_torus_experiment(
    R=R_val, r=r_val,
    paths=path_list,
    training_data=generated.numpy(),  # Pass in data points
    landmarks=[x0.cpu().numpy(), xN.cpu().numpy()],
    output_path='output/torus3_experiment_gen_el2.png',
    view_elev=35, # Top-down view makes it easier to see the gap Gap
    view_azim=-80
)

In [ ]:
pathB_np = path_B.detach().cpu().numpy()
x0_np = x0.detach().cpu().numpy()
xN_np = xN.detach().cpu().numpy()

file_base = 'output/torus_st8'
plot_torus_3d(pathB_np, R_val, r_val, x0_np, xN_np, filename=file_base+'_3d.png')
plot_poloidal(pathB_np, R_val, r_val, x0_np, xN_np, filename=file_base+'_polo.png')
plot_toroidal(pathB_np, R_val, r_val, x0_np, xN_np, filename=file_base+'_tor.png')


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
solver = RGFMSolver(alpha=1.0, beta=.01, gamma=1.0, lam=1.0, score_net=model)

x0 = torch.tensor([1.2, 0.0, 0.4]).to(device)
xN = torch.tensor([-1.2, 0.0, 0.4]).to(device)
sigma = torch.tensor(0.1).to(device)
path, losses = solver.solve(x0, xN, sigma,num_points=25, num_iters=1501, lr=5e-4)

print("Optimization Done.")

print(path.shape)

path_list = [
    {
        'path': path.reshape(-1, 3).cpu().numpy(),
        'color': 'cyan',
        'sphere_step': 2
    },
]
plot_torus_experiment(
    R=R_val, r=r_val,
    paths=path_list,
    training_data=generated.numpy(), 
    landmarks=[x0.cpu().numpy(), xN.cpu().numpy()],
    output_path='output/torus3_experiment_gen_el2.png',
    view_elev=35, 
    view_azim=-80
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

def visualize_torus_score_field(model, device, R=1.2, r=0.4, sigma_val=0.15):
    """
    Visualization Torus on Score Field
    
    Two views:
    1. Poloidal Slice (X=0 plane): See tube cross-section，Determine score whether to pull points toward surface
    2. Equatorial Slice (Z=0 plane): top view，看到环形结构
    """
    model.eval()
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    
    # ==========================================
    # Panel 1: Poloidal Slice (Y-Z Plane at X=0)
    # ==========================================
    ax1 = axes[0]
    
    # # ：tube cross-section
    y = np.linspace(R - r * 2, R + r * 2, 35)  # #  Y=R 's 
    z = np.linspace(-r * 2, r * 2, 35)
    Y, Z = np.meshgrid(y, z)
    X = np.zeros_like(Y)
    
    pts = np.stack([X.flatten(), Y.flatten(), Z.flatten()], axis=1)
    pts_tensor = torch.from_numpy(pts).float().to(device)
    sigma_tensor = torch.full((pts_tensor.shape[0],), sigma_val, device=device)
    
    with torch.no_grad():
        denoised = model(pts_tensor, sigma_tensor)
        score = (denoised - pts_tensor) / (sigma_val**2)
    score_np = score.cpu().numpy()
    
    # Y-Z 分量
    U = score_np[:, 1].reshape(Y.shape)
    V = score_np[:, 2].reshape(Z.shape)
    Magnitude = np.sqrt(U**2 + V**2)
    
    # # 
    pcm = ax1.pcolormesh(Y, Z, np.log1p(Magnitude), cmap='Blues', shading='auto', alpha=0.6)
    plt.colorbar(pcm, ax=ax1, label='Log Magnitude of Score')
    
    # # 
    skip = 2
    ax1.quiver(Y[::skip, ::skip], Z[::skip, ::skip],
               U[::skip, ::skip], V[::skip, ::skip],
               color='darkred', alpha=0.8, scale=800, width=0.004)
    
    # # tube cross-section ( (R, 0))
    theta = np.linspace(0, 2*np.pi, 100)
    tube_y = R + r * np.cos(theta)
    tube_z = r * np.sin(theta)
    ax1.plot(tube_y, tube_z, 'k-', lw=3, label='Torus Surface')
    ax1.scatter([R], [0], c='gold', s=100, marker='x', zorder=10, label='Tube Center')
    
    ax1.set_xlabel('Y Position')
    ax1.set_ylabel('Z Position')
    ax1.set_title(f'Poloidal Slice (X=0, σ={sigma_val})\nCross-section of the tube')
    ax1.set_aspect('equal')
    ax1.legend(loc='upper right', framealpha=0.5, edgecolor='none', fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # ==========================================
    # Panel 2: Equatorial Slice (X-Y Plane at Z=0)
    # ==========================================
    ax2 = axes[1]
    
    # # ：
    limit = R + r * 2
    x = np.linspace(-limit, limit, 40)
    y = np.linspace(-limit, limit, 40)
    X2, Y2 = np.meshgrid(x, y)
    Z2 = np.zeros_like(X2)
    
    pts2 = np.stack([X2.flatten(), Y2.flatten(), Z2.flatten()], axis=1)
    pts2_tensor = torch.from_numpy(pts2).float().to(device)
    sigma2_tensor = torch.full((pts2_tensor.shape[0],), sigma_val, device=device)
    
    with torch.no_grad():
        denoised2 = model(pts2_tensor, sigma2_tensor)
        score2 = (denoised2 - pts2_tensor) / (sigma_val**2)
    score2_np = score2.cpu().numpy()
    
    # X-Y 分量
    U2 = score2_np[:, 0].reshape(X2.shape)
    V2 = score2_np[:, 1].reshape(Y2.shape)
    Magnitude2 = np.sqrt(U2**2 + V2**2)
    
    # # 
    pcm2 = ax2.pcolormesh(X2, Y2, np.log1p(Magnitude2), cmap='Greens', shading='auto', alpha=0.6)
    plt.colorbar(pcm2, ax=ax2, label='Log Magnitude of Score')
    
    # # 
    skip2 = 2
    ax2.quiver(X2[::skip2, ::skip2], Y2[::skip2, ::skip2],
               U2[::skip2, ::skip2], V2[::skip2, ::skip2],
               color='darkblue', alpha=0.8, scale=800, width=0.004)
    
    # #  torus  Z=0 plane's  ()
    theta_ring = np.linspace(0, 2*np.pi, 100)
    ax2.plot((R + r) * np.cos(theta_ring), (R + r) * np.sin(theta_ring), 'k-', lw=2, label='Outer Edge')
    ax2.plot((R - r) * np.cos(theta_ring), (R - r) * np.sin(theta_ring), 'k--', lw=2, label='Inner Edge')
    ax2.plot(R * np.cos(theta_ring), R * np.sin(theta_ring), 'gray', ls=':', lw=1.5, label='Tube Center Line')
    
    ax2.set_xlabel('X Position')
    ax2.set_ylabel('Y Position')
    ax2.set_title(f'Equatorial Slice (Z=0, σ={sigma_val})\nTop-down view')
    ax2.set_aspect('equal')
    ax2.legend(loc='upper right', framealpha=0.5, edgecolor='none', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.suptitle(f'Torus Score Field Visualization (R={R}, r={r})', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return fig


def visualize_torus_score_3d(model, device, R=1.2, r=0.4, sigma_val=0.15):
    """
    更干净's  3D Score Field Visualization
    - 结构化sampling（网格而非随机）
    - 去掉坐标轴
    - 只在一个截面上显示箭头，避免视觉混乱
    """
    model.eval()
    
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # 1. 画 torus 表面
    theta = np.linspace(0, 2*np.pi, 60)
    phi = np.linspace(0, 2*np.pi, 30)
    THETA, PHI = np.meshgrid(theta, phi)
    X_s = (R + r * np.cos(PHI)) * np.cos(THETA)
    Y_s = (R + r * np.cos(PHI)) * np.sin(THETA)
    Z_s = r * np.sin(PHI)
    
    ax.plot_surface(X_s, Y_s, Z_s, color='lightgray', alpha=0.25,
                    rstride=2, cstride=2, edgecolor='none')
    
    # 2. 只在一个 poloidal 截面上sampling (比如 theta=0, 即 Y=0 plane)
    # #    ，
    n_phi = 12  # # 
    n_radial = 4  # #  ()
    
    phi_samples = np.linspace(0, 2*np.pi, n_phi, endpoint=False)
    radial_offsets = np.linspace(-r*0.6, r*0.6, n_radial)
    
    # #  toroidal 
    theta_slices = np.linspace(0, 2*np.pi, 6, endpoint=False)
    
    all_pts = []
    all_offsets = []
    
    for theta_val in theta_slices:
        for phi_val in phi_samples:
            for dr in radial_offsets:
                r_eff = r + dr
                x = (R + r_eff * np.cos(phi_val)) * np.cos(theta_val)
                y = (R + r_eff * np.cos(phi_val)) * np.sin(theta_val)
                z = r_eff * np.sin(phi_val)
                all_pts.append([x, y, z])
                all_offsets.append(dr)
    
    pts = np.array(all_pts)
    radial_offset = np.array(all_offsets)
    
    pts_tensor = torch.from_numpy(pts).float().to(device)
    sigma_tensor = torch.full((pts_tensor.shape[0],), sigma_val, device=device)
    
    # 3. compute score
    with torch.no_grad():
        denoised = model(pts_tensor, sigma_tensor)
        score = (denoised - pts_tensor) / (sigma_val**2)
    score_np = score.cpu().numpy()
    
    # # 
    magnitude = np.linalg.norm(score_np, axis=1, keepdims=True)
    arrow_length = 0.08
    score_normalized = score_np / (magnitude + 1e-8) * arrow_length
    
    # # ：，
    norm_offset = (radial_offset - radial_offset.min()) / (radial_offset.max() - radial_offset.min() + 1e-8)
    colors = plt.cm.coolwarm(norm_offset)
    
    # 4. 画箭头
    ax.quiver(pts[:, 0], pts[:, 1], pts[:, 2],
              score_normalized[:, 0], score_normalized[:, 1], score_normalized[:, 2],
              colors=colors, alpha=0.9, arrow_length_ratio=0.3, linewidth=1.2)
    
    # 5. 清理视图
    limit = R + r + 0.2
    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)
    ax.set_zlim(-limit*0.6, limit*0.6)
    ax.set_box_aspect([1, 1, 0.6])
    ax.axis('off')  # # 
    ax.view_init(elev=25, azim=-50)
    
    ax.set_title(f'Score Field on Torus (σ={sigma_val})', fontsize=14, pad=20)
    
    plt.tight_layout()
    plt.show()
    
    return fig

# ==========================================
# # Visualization
# ==========================================
# #  model, device, R_val, r_val 
fig =visualize_torus_score_field(model, device, R=R_val, r=r_val, sigma_val=0.10)
fig.savefig('output/torus_score_twoview.pdf', bbox_inches='tight')
visualize_torus_score_3d(model, device, R=R_val, r=r_val, sigma_val=0.10)